# ULMFit: Universal Language Model Fine-tuning

**Objective**: Demonstrate the ULMFit approach to transfer learning for text classification

**Paper**: Howard & Ruder (2018) - "Universal Language Model Fine-tuning for Text Classification"

**Key Innovation**: ULMFit showed that transfer learning could work effectively for NLP tasks, paving the way for BERT and modern transformers.

---

## ULMFit's 3-Step Process

1. **General-Domain LM Pre-training**: Train a language model on a large general corpus (e.g., Wikipedia)
2. **Target Task LM Fine-tuning**: Fine-tune the LM on target task data (domain adaptation)
3. **Target Task Classifier**: Add classifier and fine-tune on labeled data

## Key Techniques

- **Discriminative fine-tuning**: Different learning rates for different layers
- **Slanted triangular learning rates**: Gradually increase then decrease LR
- **Gradual unfreezing**: Unfreeze layers progressively from top to bottom

---

## Setup & Imports

In [ ]:
# Install required libraries for ULMFit implementation
# Note: These commands are commented out - uncomment if needed
# !pip install torch torchtext spacy scikit-learn matplotlib seaborn tqdm
# !python -m spacy download en_core_web_sm  # English language model for spaCy

In [ ]:
# Import all required libraries
import torch  # PyTorch for deep learning
import torch.nn as nn  # Neural network modules
import torch.nn.functional as F  # Activation functions and utilities
from torch.utils.data import Dataset, DataLoader  # Data handling utilities

import numpy as np  # Numerical computing
import pandas as pd  # Data manipulation
from sklearn.model_selection import train_test_split  # Split data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Evaluation metrics

import matplotlib.pyplot as plt  # Plotting
import seaborn as sns  # Statistical visualization
from tqdm.auto import tqdm  # Progress bars

import spacy  # NLP library for tokenization
import random  # Random number generation
import time  # Time measurement
from collections import Counter  # Count token frequencies
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Set random seeds for reproducibility
# This ensures experiments produce consistent results across runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True  # Make CUDA operations deterministic

# Determine device (GPU if available, otherwise CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Load Dataset

We'll use the **IMDb movie reviews** dataset for sentiment analysis.

For demonstration purposes, we'll use a subset to speed up training.

In [ ]:
# Load the IMDb movie review dataset
from datasets import load_dataset

print("Loading IMDb dataset...")
dataset = load_dataset('imdb')

# Use subset for faster experimentation
# In practice, you would use larger datasets for better results
TRAIN_SIZE = 5000   # Reviews for training the classifier
TEST_SIZE = 1000    # Reviews for testing the classifier
LM_SIZE = 10000     # Reviews for language model fine-tuning (unlabeled)

# Sample data randomly with fixed seed for reproducibility
train_data = dataset['train'].shuffle(seed=SEED).select(range(TRAIN_SIZE))
test_data = dataset['test'].shuffle(seed=SEED).select(range(TEST_SIZE))
lm_data = dataset['train'].shuffle(seed=SEED).select(range(LM_SIZE))

# Display dataset statistics
print(f"\nDataset sizes:")
print(f"  Language Model data: {len(lm_data)} samples")
print(f"  Classification train: {len(train_data)} samples")
print(f"  Classification test: {len(test_data)} samples")

# Show an example to understand data format
print(f"\nExample review:")
print(f"Text: {train_data[0]['text'][:300]}...")
print(f"Label: {train_data[0]['label']} (0=negative, 1=positive)")

## Tokenization & Vocabulary

We'll use spaCy for tokenization and build our own vocabulary.

In [ ]:
# Load spaCy tokenizer for English
# We disable parser, NER, and tagger to speed up tokenization (we only need tokens)
nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner', 'tagger'])

def tokenize(text):
    """
    Tokenize text using spaCy and convert to lowercase.
    
    Args:
        text: Input string to tokenize
    
    Returns:
        List of lowercase tokens
    """
    return [token.text.lower() for token in nlp(text)]

# Test tokenization on a sample sentence
sample = "This movie was absolutely fantastic!"
print(f"Original: {sample}")
print(f"Tokenized: {tokenize(sample)}")
print("\n💡 Tokenization splits text into words and punctuation marks")

In [ ]:
# Build vocabulary from the language model data
print("Building vocabulary...")

# Count frequency of each token across all texts
counter = Counter()
for example in tqdm(lm_data, desc="Tokenizing"):
    tokens = tokenize(example['text'])
    counter.update(tokens)

# Create vocabulary with minimum frequency threshold
# This filters out rare words that might be typos or very uncommon
MIN_FREQ = 5  # Only include words that appear at least 5 times
vocab = ['<pad>', '<unk>', '<bos>', '<eos>']  # Special tokens first

# Add words that meet the minimum frequency threshold
vocab += [word for word, freq in counter.items() if freq >= MIN_FREQ]

# Create mappings: word ↔ index
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f"\nVocabulary size: {len(vocab)}")
print(f"Most common words: {counter.most_common(10)}")

# Special token indices (used for padding, unknown words, etc.)
PAD_IDX = word2idx['<pad>']   # Padding token for sequences of different lengths
UNK_IDX = word2idx['<unk>']   # Unknown token for words not in vocabulary
BOS_IDX = word2idx['<bos>']   # Beginning of sequence token
EOS_IDX = word2idx['<eos>']   # End of sequence token

print(f"\n💡 Vocabulary contains {len(vocab):,} words")
print(f"   Most common words are function words: 'the', 'a', 'and', etc.")

## AWD-LSTM Language Model Architecture

ULMFit uses **AWD-LSTM** (ASGD Weight-Dropped LSTM):
- Multi-layer LSTM with dropout
- Weight dropping (dropout on hidden-to-hidden connections)
- Embedding dropout, input dropout, output dropout

In [ ]:
class AWD_LSTM_LM(nn.Module):
    """
    AWD-LSTM Language Model (simplified version of ULMFit architecture).
    
    AWD-LSTM = ASGD Weight-Dropped LSTM
    Key features:
    - Multiple LSTM layers for capturing complex patterns
    - Dropout for regularization (prevents overfitting)
    - Weight tying: decoder uses same weights as encoder (reduces parameters)
    
    This model is trained to predict the next word given previous words.
    """

    def __init__(self, vocab_size, emb_dim=400, hidden_dim=1152, num_layers=3,
                 dropout=0.4, tie_weights=True):
        """
        Args:
            vocab_size: Number of words in vocabulary
            emb_dim: Dimension of word embeddings (vector representation)
            hidden_dim: Size of LSTM hidden state
            num_layers: Number of stacked LSTM layers
            dropout: Dropout probability (0.4 = 40% of neurons dropped)
            tie_weights: Whether to tie encoder and decoder weights
        """
        super().__init__()

        self.vocab_size = vocab_size
        self.emb_dim = emb_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding layer: converts word indices to dense vectors
        self.encoder = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD_IDX)
        self.emb_dropout = nn.Dropout(dropout)

        # LSTM layers: learn sequential patterns in text
        self.lstm = nn.LSTM(
            emb_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0,  # Dropout between LSTM layers
            batch_first=True  # Input shape: [batch_size, seq_len, features]
        )

        # Output dropout
        self.output_dropout = nn.Dropout(dropout)

        # Decoder: projects LSTM output back to vocabulary size
        self.decoder = nn.Linear(hidden_dim, vocab_size)

        # Weight tying: share weights between encoder and decoder
        # This reduces model size and often improves performance
        if tie_weights:
            self.decoder.weight = self.encoder.weight

        self.init_weights()

    def init_weights(self):
        """Initialize weights with small random values"""
        init_range = 0.1
        self.encoder.weight.data.uniform_(-init_range, init_range)
        self.decoder.bias.data.zero_()
        self.decoder.weight.data.uniform_(-init_range, init_range)

    def forward(self, x, hidden=None):
        """
        Forward pass through the language model.
        
        Args:
            x: Input token indices [batch_size, seq_len]
            hidden: LSTM hidden state (tuple of h, c)
        
        Returns:
            output: Predicted word scores [batch_size, seq_len, vocab_size]
            hidden: Updated LSTM hidden state
        """
        # Convert token indices to embeddings
        embedded = self.encoder(x)  # [batch, seq_len, emb_dim]
        embedded = self.emb_dropout(embedded)

        # Process through LSTM
        output, hidden = self.lstm(embedded, hidden)
        output = self.output_dropout(output)

        # Project to vocabulary size
        decoded = self.decoder(output)  # [batch, seq_len, vocab_size]

        return decoded, hidden

    def init_hidden(self, batch_size):
        """
        Initialize LSTM hidden state with zeros.
        
        Returns:
            Tuple of (h, c) where both are [num_layers, batch_size, hidden_dim]
        """
        weight = next(self.parameters())
        return (weight.new_zeros(self.num_layers, batch_size, self.hidden_dim),
                weight.new_zeros(self.num_layers, batch_size, self.hidden_dim))

# Test the model architecture
test_model = AWD_LSTM_LM(len(vocab), emb_dim=200, hidden_dim=400, num_layers=2)
print(f"\nModel parameters: {sum(p.numel() for p in test_model.parameters()):,}")
print(f"\n💡 This model learns to predict the next word in a sequence")
print(f"   It will be trained on unlabeled text (no sentiment labels needed!)")
print(f"\nModel architecture:\n{test_model}")

## Dataset Classes for Language Model

In [ ]:
class LMDataset(Dataset):
    """
    Dataset for Language Model training.
    
    Converts text to sequences of token indices.
    Each example is a fixed-length sequence where the target
    is the same sequence shifted by one position.
    """

    def __init__(self, texts, word2idx, seq_len=70):
        """
        Args:
            texts: List of text strings
            word2idx: Dictionary mapping words to indices
            seq_len: Length of each sequence
        """
        self.word2idx = word2idx
        self.seq_len = seq_len

        # Tokenize all texts and convert to indices
        print("Tokenizing texts...")
        all_tokens = []
        for text in tqdm(texts):
            tokens = tokenize(text)
            # Convert tokens to indices (use UNK_IDX for unknown words)
            indices = [word2idx.get(token, UNK_IDX) for token in tokens]
            all_tokens.extend(indices)

        self.data = torch.tensor(all_tokens, dtype=torch.long)
        print(f"Total tokens: {len(self.data):,}")

    def __len__(self):
        """Number of sequences in dataset"""
        return len(self.data) // self.seq_len - 1

    def __getitem__(self, idx):
        """
        Get a sequence and its target.
        
        Returns:
            x: Input sequence [seq_len]
            y: Target sequence [seq_len] (same as x but shifted by 1)
        """
        start_idx = idx * self.seq_len
        end_idx = start_idx + self.seq_len

        # Input: tokens at positions [start:end]
        x = self.data[start_idx:end_idx]
        # Target: tokens at positions [start+1:end+1] (next word prediction)
        y = self.data[start_idx + 1:end_idx + 1]

        return x, y

# Create dataset for language model training
lm_texts = [example['text'] for example in lm_data]
lm_dataset = LMDataset(lm_texts, word2idx, seq_len=70)
lm_loader = DataLoader(lm_dataset, batch_size=64, shuffle=True, drop_last=True)

print(f"\nLanguage Model batches: {len(lm_loader)}")
print(f"\n💡 Each batch contains 64 sequences of 70 tokens")
print(f"   The model learns to predict token i+1 given tokens 1 to i")

---

# Step 1: General-Domain LM Pre-training

**Note**: In practice, you would use a pre-trained model trained on Wikipedia (WikiText-103).

For this demo, we'll **simulate** this step by initializing a model (imagine it's already been trained on WikiText).

In [ ]:
print("=" * 60)
print("STEP 1: GENERAL-DOMAIN LM PRE-TRAINING (simulated)")
print("=" * 60)

# In real ULMFit, this step trains the language model on a large corpus
# like Wikipedia (WikiText-103 dataset with 103 million words)
# This teaches the model general language understanding

# For this demo, we initialize a model (pretending it's pre-trained)
# In practice, you would load pre-trained weights here
pretrained_lm = AWD_LSTM_LM(
    vocab_size=len(vocab),
    emb_dim=400,
    hidden_dim=1152,
    num_layers=3,
    dropout=0.4,
    tie_weights=False  # Set to False to avoid dimension mismatch in this demo
).to(device)

print(f"\n✅ 'Pre-trained' Language Model initialized")
print(f"   Parameters: {sum(p.numel() for p in pretrained_lm.parameters()):,}")
print(f"\n📝 In real ULMFit: this model would be trained on 103M words from Wikipedia")
print(f"   Training time: ~24 hours on GPU")
print(f"   Result: Model learns general language patterns")
print(f"\n💡 This is analogous to BERT's pre-training phase")
print(f"   The model learns:")
print(f"   - Grammar and syntax")
print(f"   - Word meanings and relationships")
print(f"   - General world knowledge")

---

# Step 2: Target Task LM Fine-tuning

**Objective**: Adapt the language model to the target domain (movie reviews)

**Key technique**: Fine-tune on **unlabeled** target task data to learn domain-specific language patterns.

In [ ]:
def train_lm_epoch(model, dataloader, optimizer, device, clip=0.25):
    """
    Train language model for one epoch.
    
    Args:
        model: Language model to train
        dataloader: DataLoader providing batches
        optimizer: Optimization algorithm
        device: CPU or CUDA device
        clip: Gradient clipping value (prevents exploding gradients)
    
    Returns:
        Average loss over all batches
    """
    model.train()
    total_loss = 0

    for x, y in tqdm(dataloader, desc="Training LM"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        # Forward pass: predict next words
        output, _ = model(x)

        # Reshape for loss calculation
        output = output.view(-1, output.size(-1))  # [batch*seq_len, vocab_size]
        y = y.view(-1)  # [batch*seq_len]

        # Calculate cross-entropy loss (measures prediction error)
        loss = F.cross_entropy(output, y, ignore_index=PAD_IDX)

        # Backward pass: compute gradients
        loss.backward()

        # Gradient clipping: prevents gradients from becoming too large
        # This is important for training RNNs which can have exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate_lm(model, dataloader, device):
    """
    Evaluate language model.
    
    Returns:
        Average loss on the evaluation set
    """
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            output, _ = model(x)
            output = output.view(-1, output.size(-1))
            y = y.view(-1)

            loss = F.cross_entropy(output, y, ignore_index=PAD_IDX)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
print("=" * 60)
print("STEP 2: TARGET TASK LM FINE-TUNING")
print("=" * 60)
print("\n💡 Goal: Adapt the general language model to movie review language")
print("   This helps the model understand domain-specific vocabulary and style")
print("   Example: 'plot', 'acting', 'cinematography' are important in movie reviews")

# Fine-tune the language model on movie reviews (unlabeled data)
finetuned_lm = pretrained_lm  # Start from 'pre-trained' model

# Discriminative fine-tuning: different learning rates for each layer
# ULMFit paper recommends: each lower layer uses lr/2.6 of the layer above
# This is because:
# - Lower layers learn general features (don't change much)
# - Higher layers learn task-specific features (need more adaptation)
lr = 1e-3
optimizer = torch.optim.Adam(finetuned_lm.parameters(), lr=lr)

# Train for a few epochs
LM_EPOCHS = 2  # Increase for better results (but takes longer)
lm_losses = []

print(f"\nFine-tuning LM on {len(lm_dataset):,} sequences...")
print(f"   Training on unlabeled movie reviews (no sentiment labels needed!)")
start_time = time.time()

for epoch in range(LM_EPOCHS):
    print(f"\n📍 Epoch {epoch + 1}/{LM_EPOCHS}")
    loss = train_lm_epoch(finetuned_lm, lm_loader, optimizer, device)
    lm_losses.append(loss)
    perplexity = np.exp(loss)  # Perplexity: measure of prediction uncertainty
    print(f"   Loss: {loss:.4f} | Perplexity: {perplexity:.2f}")

lm_time = time.time() - start_time

print(f"\n✅ LM Fine-tuning complete in {lm_time:.2f}s")
print(f"   Final perplexity: {np.exp(lm_losses[-1]):.2f}")
print(f"\n💡 Perplexity measures how 'surprised' the model is by the text")
print(f"   Lower perplexity = better language understanding")
print(f"   The model now understands movie review language better!")

## Test Language Model

Let's see if the LM can generate movie review-like text!

In [ ]:
def generate_text(model, word2idx, idx2word, start_text="this movie", max_len=50, temperature=0.8):
    """
    Generate text using the language model.
    
    Args:
        model: Trained language model
        word2idx: Word to index mapping
        idx2word: Index to word mapping
        start_text: Initial text to start generation
        max_len: Maximum number of tokens to generate
        temperature: Controls randomness (higher = more random)
    
    Returns:
        Generated text string
    """
    model.eval()

    # Tokenize starting text
    tokens = tokenize(start_text)
    indices = [word2idx.get(token, UNK_IDX) for token in tokens]

    generated = indices.copy()

    with torch.no_grad():
        hidden = None

        for _ in range(max_len):
            # Get last token as input
            x = torch.tensor([generated[-1]], dtype=torch.long).unsqueeze(0).to(device)

            # Predict next token
            output, hidden = model(x, hidden)

            # Apply temperature and sample from probability distribution
            # Temperature controls randomness:
            # - High temperature (>1): more random, diverse text
            # - Low temperature (<1): more predictable, safe text
            probs = F.softmax(output[0, -1] / temperature, dim=0)
            next_idx = torch.multinomial(probs, 1).item()

            generated.append(next_idx)

            # Stop at end of sentence token
            if next_idx == EOS_IDX:
                break

    # Convert indices back to text
    generated_text = ' '.join([idx2word.get(idx, '<unk>') for idx in generated])
    return generated_text

# Generate text examples to see what the model learned
print("🎬 Generated movie reviews:\n")
print("💡 This demonstrates that the model learned movie review language patterns\n")
prompts = ["this movie", "the acting", "i loved", "terrible film"]

for prompt in prompts:
    generated = generate_text(finetuned_lm, word2idx, idx2word, start_text=prompt, max_len=30)
    print(f"Prompt: '{prompt}'")
    print(f"Generated: {generated}")
    print()

print("📝 Notice: The generated text sounds like movie reviews!")
print("   The model learned domain-specific vocabulary and sentence structures")

---

# Step 3: Target Task Classifier

**Objective**: Add a classification head and fine-tune on labeled data

**Key techniques**:
1. **Gradual unfreezing**: Start with frozen layers, unfreeze progressively
2. **Discriminative fine-tuning**: Different learning rates per layer
3. **Slanted triangular learning rates**: Linearly increase then decay

In [ ]:
class ULMFitClassifier(nn.Module):
    """
    ULMFit Classifier: Language Model + Classification Head.
    
    This model consists of:
    1. Fine-tuned language model (encoder + LSTM)
    2. Classification head (fully connected layers)
    
    The LM part provides rich text representations,
    and the classifier learns to map them to sentiment labels.
    """

    def __init__(self, lm_model, num_classes=2, hidden_dim=50):
        """
        Args:
            lm_model: Pre-trained/fine-tuned language model
            num_classes: Number of output classes (2 for binary sentiment)
            hidden_dim: Size of hidden layer in classifier head
        """
        super().__init__()

        # Reuse components from the language model
        self.encoder = lm_model.encoder  # Word embeddings
        self.lstm = lm_model.lstm  # LSTM layers
        self.emb_dropout = lm_model.emb_dropout
        self.output_dropout = lm_model.output_dropout

        lstm_hidden_dim = lm_model.hidden_dim

        # NEW: Classification head
        # This is a small neural network on top of the LSTM
        self.fc1 = nn.Linear(lstm_hidden_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.4)

    def forward(self, x):
        """
        Forward pass for classification.
        
        Args:
            x: Input token indices [batch_size, seq_len]
        
        Returns:
            logits: Class scores [batch_size, num_classes]
        """
        # Get word embeddings
        embedded = self.encoder(x)
        embedded = self.emb_dropout(embedded)

        # Process through LSTM
        output, (hidden, cell) = self.lstm(embedded)
        output = self.output_dropout(output)

        # Use last hidden state for classification
        # This vector represents the entire input sequence
        last_hidden = hidden[-1]  # [batch_size, hidden_dim]

        # Pass through classification head
        x = self.dropout(last_hidden)
        x = F.relu(self.fc1(x))  # First layer with ReLU activation
        x = self.dropout(x)
        logits = self.fc2(x)  # Output layer (no activation)

        return logits

# Create classifier from fine-tuned language model
ulmfit_classifier = ULMFitClassifier(finetuned_lm, num_classes=2).to(device)

print("\n🏗️ ULMFit Classifier created")
print(f"   Total parameters: {sum(p.numel() for p in ulmfit_classifier.parameters()):,}")
print(f"   Trainable parameters: {sum(p.numel() for p in ulmfit_classifier.parameters() if p.requires_grad):,}")
print(f"\n💡 The classifier reuses the fine-tuned language model")
print(f"   Only the classification head (fc1, fc2) is new")
print(f"   This allows leveraging all the language knowledge learned in Steps 1 & 2!")

## Dataset for Classification

In [ ]:
class ClassificationDataset(Dataset):
    """
    Dataset for sentiment classification.
    
    Converts text and labels to token indices.
    """

    def __init__(self, texts, labels, word2idx, max_len=200):
        """
        Args:
            texts: List of text strings
            labels: List of class labels (0 or 1)
            word2idx: Dictionary mapping words to indices
            max_len: Maximum sequence length (truncate longer sequences)
        """
        self.word2idx = word2idx
        self.max_len = max_len

        self.data = []
        for text, label in zip(texts, labels):
            tokens = tokenize(text)
            # Convert to indices and truncate if needed
            indices = [word2idx.get(token, UNK_IDX) for token in tokens[:max_len]]
            self.data.append((indices, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    """
    Collate function for DataLoader.
    
    Pads sequences to the same length within each batch.
    """
    # Sort by length (descending) - helps with RNN efficiency
    batch.sort(key=lambda x: len(x[0]), reverse=True)

    sequences, labels = zip(*batch)

    # Pad all sequences to the length of the longest sequence
    max_len = len(sequences[0])
    padded = [seq + [PAD_IDX] * (max_len - len(seq)) for seq in sequences]

    return torch.tensor(padded, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

# Create classification datasets
train_texts = [ex['text'] for ex in train_data]
train_labels = [ex['label'] for ex in train_data]
test_texts = [ex['text'] for ex in test_data]
test_labels = [ex['label'] for ex in test_data]

train_clf_dataset = ClassificationDataset(train_texts, train_labels, word2idx)
test_clf_dataset = ClassificationDataset(test_texts, test_labels, word2idx)

train_clf_loader = DataLoader(train_clf_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_clf_loader = DataLoader(test_clf_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)

print(f"Classification datasets created")
print(f"  Train batches: {len(train_clf_loader)}")
print(f"  Test batches: {len(test_clf_loader)}")
print(f"\n💡 Now we'll train the classifier on labeled sentiment data")

In [ ]:
def train_classifier_epoch(model, dataloader, optimizer, device):
    """
    Train classifier for one epoch.
    
    Args:
        model: ULMFit classifier
        dataloader: DataLoader with labeled data
        optimizer: Optimization algorithm
        device: CPU or CUDA device
    
    Returns:
        Average loss and accuracy for the epoch
    """
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(dataloader, desc="Training Classifier"):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        # Forward pass: get class predictions
        logits = model(x)
        loss = F.cross_entropy(logits, y)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Calculate metrics
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

def evaluate_classifier(model, dataloader, device):
    """
    Evaluate classifier on test data.
    
    Returns:
        Accuracy, predictions, and true labels
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    return accuracy, all_preds, all_labels

In [ ]:
print("=" * 60)
print("STEP 3: CLASSIFIER TRAINING WITH GRADUAL UNFREEZING")
print("=" * 60)
print("\n💡 ULMFit uses 'gradual unfreezing' - a key technique:")
print("   Phase 1: Train only the classifier head (freeze language model)")
print("   Phase 2: Unfreeze all layers and fine-tune with different learning rates")
print("   This prevents catastrophic forgetting of the language model's knowledge")

# Phase 1: Train only the classifier head (freeze LM)
print("\n🔒 Phase 1: Training classifier head (LM frozen)")
print("   Only the new classification layers (fc1, fc2) are trained")
print("   The language model stays frozen to preserve its knowledge")

# Freeze language model layers
for param in ulmfit_classifier.encoder.parameters():
    param.requires_grad = False
for param in ulmfit_classifier.lstm.parameters():
    param.requires_grad = False

# Optimizer trains only unfrozen parameters
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, ulmfit_classifier.parameters()),
    lr=1e-3
)

# Train for 1 epoch
loss, acc = train_classifier_epoch(ulmfit_classifier, train_clf_loader, optimizer, device)
print(f"  Loss: {loss:.4f} | Accuracy: {acc:.4f}")

# Phase 2: Unfreeze all and fine-tune with discriminative learning rates
print("\n🔓 Phase 2: Fine-tuning entire model (gradual unfreezing)")
print("   All layers are now trainable")
print("   Discriminative learning rates: different LR for each layer group")
print("   Lower layers (general features) → smaller LR")
print("   Higher layers (task-specific) → larger LR")

# Unfreeze all layers
for param in ulmfit_classifier.parameters():
    param.requires_grad = True

# Discriminative fine-tuning: assign different learning rates to layer groups
# This is a key ULMFit technique!
# - Embeddings: smallest LR (most general, least task-specific)
# - LSTM: medium LR
# - Classifier head: largest LR (most task-specific)
base_lr = 1e-4
optimizer = torch.optim.Adam([
    {'params': ulmfit_classifier.encoder.parameters(), 'lr': base_lr / 2.6},
    {'params': ulmfit_classifier.lstm.parameters(), 'lr': base_lr},
    {'params': ulmfit_classifier.fc1.parameters(), 'lr': base_lr * 2.6},
    {'params': ulmfit_classifier.fc2.parameters(), 'lr': base_lr * 2.6}
])

# Train for several epochs
CLF_EPOCHS = 3
train_losses = []
train_accs = []

start_time = time.time()

for epoch in range(CLF_EPOCHS):
    print(f"\n📍 Epoch {epoch + 1}/{CLF_EPOCHS}")
    loss, acc = train_classifier_epoch(ulmfit_classifier, train_clf_loader, optimizer, device)
    train_losses.append(loss)
    train_accs.append(acc)
    print(f"  Loss: {loss:.4f} | Accuracy: {acc:.4f}")

clf_time = time.time() - start_time

# Final evaluation on test set
test_acc, test_preds, test_labels = evaluate_classifier(ulmfit_classifier, test_clf_loader, device)

print(f"\n✅ ULMFit Training Complete")
print(f"   Training time: {clf_time:.2f}s")
print(f"   Final train accuracy: {train_accs[-1]:.4f}")
print(f"   Test accuracy: {test_acc:.4f}")
print(f"\n💡 The model successfully learned sentiment classification")
print(f"   thanks to transfer learning from the language model!")

---

# Results & Visualization

In [ ]:
# Classification Report: detailed performance metrics
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(test_labels, test_preds, target_names=['Negative', 'Positive']))

print("\n📊 Understanding the metrics:")
print("   - Precision: When we predict a class, how often are we correct?")
print("   - Recall: Of all examples in a class, how many did we find?")
print("   - F1-score: Harmonic mean of precision and recall")
print("   - Support: Number of examples in each class")
print("\n💡 A balanced model should have similar scores for both classes")

In [ ]:
# Visualizations to understand model performance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Training accuracy over epochs
axes[0].plot(range(1, CLF_EPOCHS + 1), train_accs, 'o-', color='#27ae60', linewidth=2, markersize=8)
axes[0].axhline(y=test_acc, color='#e74c3c', linestyle='--', label=f'Test Acc: {test_acc:.3f}')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('ULMFit Training Progress', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Training loss over epochs
axes[1].plot(range(1, CLF_EPOCHS + 1), train_losses, 'o-', color='#3498db', linewidth=2, markersize=8)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Loss', fontsize=12)
axes[1].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 3. Confusion Matrix
# Shows how many examples were correctly/incorrectly classified
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
axes[2].set_xlabel('Predicted', fontsize=12)
axes[2].set_ylabel('True', fontsize=12)
axes[2].set_title('Confusion Matrix', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n📈 Visualization insights:")
print("   - Training accuracy improves across epochs (learning is happening!)")
print("   - Training loss decreases (model gets better at predicting)")
print("   - Confusion matrix shows the distribution of correct/incorrect predictions")
print("   - Diagonal elements (top-left, bottom-right) are correct predictions")

## Test on Custom Examples

In [ ]:
def predict_sentiment(model, text, word2idx, device):
    """
    Predict sentiment for a single text input.
    
    Args:
        model: Trained ULMFit classifier
        text: Input text string
        word2idx: Word to index mapping
        device: CPU or CUDA device
    
    Returns:
        sentiment: "Positive" or "Negative"
        confidence: Probability of predicted class (0-1)
    """
    model.eval()

    # Tokenize and convert to indices
    tokens = tokenize(text)
    indices = [word2idx.get(token, UNK_IDX) for token in tokens]

    # Convert to tensor
    x = torch.tensor([indices], dtype=torch.long).to(device)

    # Predict
    with torch.no_grad():
        logits = model(x)
        probs = F.softmax(logits, dim=1)  # Convert logits to probabilities
        pred = torch.argmax(probs, dim=1).item()  # Get predicted class
        confidence = probs[0, pred].item()  # Confidence in prediction

    sentiment = "Positive" if pred == 1 else "Negative"
    return sentiment, confidence

# Test on custom examples
test_examples = [
    "This movie was absolutely fantastic! I loved every minute of it.",
    "Terrible film. Waste of time and money. Avoid at all costs.",
    "It was okay, nothing special but not terrible either.",
    "The acting was superb and the plot kept me on the edge of my seat!",
    "Boring and predictable. I fell asleep halfway through."
]

print("\n🎬 Testing ULMFit on custom examples:\n")
print("=" * 80)
print("💡 These are examples the model has never seen before!\n")

for text in test_examples:
    sentiment, confidence = predict_sentiment(ulmfit_classifier, text, word2idx, device)
    print(f"Text: {text}")
    print(f"Prediction: {sentiment} (confidence: {confidence:.2%})")
    print("-" * 80)

print("\n📝 Key Takeaways:")
print("   - ULMFit successfully performs sentiment analysis")
print("   - Transfer learning (3-step process) enables good performance with limited data")
print("   - The language model provides a strong foundation for the classifier")
print("   - Gradual unfreezing and discriminative learning rates are crucial techniques")


---

**👨‍🏫 Andrea Belli - Text Mining Course - AA 2025/2026**